# 🎓 SyllaBot — Checkpoint 1: Text Preprocessing Pipeline
**IS Professional Elective #4 — Generative AI Systems**  
**Group: Sugar Group | A.Y. 2026–2027**

This notebook covers **Deliverable #3** of Checkpoint 1:
- Load PDF and DOCX syllabus files
- Clean and normalize raw text
- Tokenize the processed text
- Show before/after examples


## Step 1 — Install Required Libraries

In [ ]:
# Install all required packages
!pip install pdfplumber python-docx nltk -q

## Step 2 — Upload Your Syllabus Files
Run this cell to upload your PDF and DOCX files from your computer.

In [ ]:
from google.colab import files
import os

# Create a folder to store uploaded syllabi
os.makedirs('syllabi', exist_ok=True)

print('📂 Upload your syllabus files (PDF and/or DOCX)...')
uploaded = files.upload()

# Move uploaded files into the syllabi folder
for filename in uploaded.keys():
    os.rename(filename, f'syllabi/{filename}')
    print(f'  ✅ Saved: {filename}')

print(f'\n📁 Total files uploaded: {len(uploaded)}')

## Step 3 — Extract Raw Text from Files
We extract raw text from both PDF and DOCX formats.

In [ ]:
import pdfplumber
from docx import Document

def extract_text_from_pdf(filepath):
    """Extract raw text from a PDF file."""
    text = ''
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + '\n'
    return text

def extract_text_from_docx(filepath):
    """Extract raw text from a DOCX file."""
    doc = Document(filepath)
    text = '\n'.join([para.text for para in doc.paragraphs if para.text.strip()])
    return text

# Load all files from the syllabi folder
raw_documents = {}  # {filename: raw_text}

for filename in os.listdir('syllabi'):
    filepath = f'syllabi/{filename}'
    if filename.endswith('.pdf'):
        raw_documents[filename] = extract_text_from_pdf(filepath)
        print(f'  📄 Extracted PDF: {filename} ({len(raw_documents[filename])} chars)')
    elif filename.endswith('.docx'):
        raw_documents[filename] = extract_text_from_docx(filepath)
        print(f'  📝 Extracted DOCX: {filename} ({len(raw_documents[filename])} chars)')
    else:
        print(f'  ⚠️  Skipped unsupported file: {filename}')

print(f'\n✅ Total documents loaded: {len(raw_documents)}')

## Step 4 — BEFORE Example (Raw Text Preview)
This shows what the text looks like **before** cleaning.

In [ ]:
# Pick the first document to show the before/after example
sample_filename = list(raw_documents.keys())[0]
sample_raw = raw_documents[sample_filename]

print('=' * 60)
print(f'📄 BEFORE CLEANING — {sample_filename}')
print('=' * 60)
print(sample_raw[:1000])  # Show first 1000 characters
print('...')

## Step 5 — Text Cleaning & Normalization
We remove noise: extra whitespace, special characters, headers/footers, and empty lines.

In [ ]:
import re

def clean_text(text):
    """
    Clean and normalize raw text extracted from syllabus documents.
    Steps:
      1. Remove non-ASCII characters
      2. Remove URLs
      3. Remove extra whitespace and blank lines
      4. Normalize multiple spaces into one
      5. Strip leading/trailing whitespace per line
      6. Convert to lowercase
    """
    # 1. Remove non-ASCII characters (e.g., weird PDF symbols)
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. Remove special characters except basic punctuation
    text = re.sub(r'[^\w\s.,;:\-\'"()%/]', ' ', text)

    # 4. Normalize multiple spaces into one
    text = re.sub(r' +', ' ', text)

    # 5. Remove lines that are just numbers (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 6. Strip each line and remove blank lines
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    text = '\n'.join(lines)

    # 7. Convert to lowercase
    text = text.lower()

    return text

# Apply cleaning to all documents
cleaned_documents = {}
for filename, raw_text in raw_documents.items():
    cleaned_documents[filename] = clean_text(raw_text)
    print(f'  ✅ Cleaned: {filename}')

print('\n🧹 All documents cleaned!')

## Step 6 — AFTER Example (Cleaned Text Preview)
This shows what the same text looks like **after** cleaning.

In [ ]:
sample_cleaned = cleaned_documents[sample_filename]

print('=' * 60)
print(f'✨ AFTER CLEANING — {sample_filename}')
print('=' * 60)
print(sample_cleaned[:1000])
print('...')

# Show size reduction
before_len = len(raw_documents[sample_filename])
after_len = len(sample_cleaned)
reduction = round((1 - after_len / before_len) * 100, 2)
print(f'\n📊 Size before: {before_len} chars')
print(f'📊 Size after:  {after_len} chars')
print(f'📉 Noise reduced by: {reduction}%')

## Step 7 — Tokenization
We split the cleaned text into individual word tokens using NLTK.

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    """
    Tokenize cleaned text into individual word tokens.
    Also removes stopwords (common words like 'the', 'is', 'and').
    """
    tokens = word_tokenize(text)
    # Keep only alphabetic tokens and remove stopwords
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return tokens

# Tokenize all cleaned documents
tokenized_documents = {}
for filename, cleaned_text in cleaned_documents.items():
    tokenized_documents[filename] = tokenize_text(cleaned_text)
    print(f'  🔤 Tokenized: {filename} → {len(tokenized_documents[filename])} tokens')

print('\n✅ Tokenization complete!')

## Step 8 — Tokenization Before/After Example

In [ ]:
sample_tokens = tokenized_documents[sample_filename]

print('=' * 60)
print('BEFORE TOKENIZATION (cleaned text excerpt):')
print('=' * 60)
print(sample_cleaned[:300])

print('\n' + '=' * 60)
print('AFTER TOKENIZATION (first 50 tokens):')
print('=' * 60)
print(sample_tokens[:50])
print(f'\n📊 Total unique tokens in this document: {len(set(sample_tokens))}')

## Step 9 — Summary of All Documents

In [ ]:
print('=' * 60)
print('📋 PREPROCESSING SUMMARY — ALL DOCUMENTS')
print('=' * 60)
print(f'{"File":<40} {"Raw Chars":>10} {"Clean Chars":>12} {"Tokens":>8}')
print('-' * 72)

total_tokens = 0
for filename in raw_documents:
    raw_len   = len(raw_documents[filename])
    clean_len = len(cleaned_documents[filename])
    tok_count = len(tokenized_documents[filename])
    total_tokens += tok_count
    print(f'{filename:<40} {raw_len:>10} {clean_len:>12} {tok_count:>8}')

print('-' * 72)
print(f'{"TOTAL":<40} {"": >10} {"": >12} {total_tokens:>8}')
print(f'\n✅ Pipeline complete! {len(raw_documents)} documents preprocessed.')
print(f'📦 Total tokens across all syllabi: {total_tokens:,}')

## Step 10 — Save Cleaned Text Files
Save each cleaned document as a `.txt` file for use in the next steps (embedding generation).

In [ ]:
import os

os.makedirs('syllabi_cleaned', exist_ok=True)

for filename, cleaned_text in cleaned_documents.items():
    # Save as .txt with same base name
    base_name = os.path.splitext(filename)[0]
    output_path = f'syllabi_cleaned/{base_name}.txt'
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(cleaned_text)
    print(f'  💾 Saved: {output_path}')

print('\n✅ All cleaned files saved to /syllabi_cleaned/')
print('📌 These files will be used in the next notebook (Embedding Generation).')